<div style="max-width:100%;box-sizing:border-box;border-top:4px solid #0f766e;padding:24px 0">
<div style="color:#0f766e;font-weight:700">EXTENSION · DATA WAREHOUSING WITH APACHE DORIS</div>
<h1>Module 6–7 Extension Lab: Schema Changes, Load-Based Deletion, and Conflict Detection</h1>
<p>Target Doris 4.1.3 · Independent ext_* lab tables</p>
</div>

[Extension home](README.md) · [Main course contents](../README.md)

Complete main Labs 6 and 7 first. Rebuild only `ext_schema`, `ext_delete_sign`, and `ext_event_stage`; do not modify orders_clean/current or business history.
Allow 25–35 minutes. Run only in a single kernel in the course sandbox and retain results for troubleshooting.
Validation: old values are clear after adding a column, and downstream explicit column projections remain unchanged; old versions cannot resurrect orders after deletion; different content for the same event ID is detected.
This does not implement real Binlog recovery or provide an atomic conflict rejection service for concurrent consumers.

In [ ]:
from pathlib import Path
import sys
COURSE_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))
from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect, fixture, normalized
from dw_course.ui import show_sql, show_response
lab = connect_sandbox()
lab.sql("SELECT VERSION() AS version")
lab.sql("SHOW BACKENDS")

## 1. Adding Columns, Compatibility, and Asynchronous Change Status

Copy ten orders to an independent table and first record the downstream explicit column projection. Add a nullable channel column; old records have NULL.
Then change amount_cents from INT to BIGINT, wait for the SHOW ALTER TABLE COLUMN job to finish, and verify the results.
Lightweight versus heavyweight depends on the operation and target version; do not infer from this run's duration that all Schema Changes modify only metadata.

In [ ]:
import time
lab.execute("DROP TABLE IF EXISTS ext_schema")
lab.execute('''CREATE TABLE ext_schema (order_id BIGINT, amount_cents INT)
DUPLICATE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1 PROPERTIES("replication_num"="1")''')
lab.execute("INSERT INTO ext_schema SELECT order_id,CAST(order_amount*100 AS INT) FROM orders_clean")
expected = lab.query("SELECT order_id,amount_cents FROM ext_schema ORDER BY order_id")
expect(lab.query("SELECT COUNT(*),SUM(amount_cents) FROM ext_schema"), [(10,140000)])
for statement, column, target_type in (("ALTER TABLE ext_schema ADD COLUMN order_channel VARCHAR(20) NULL", "order_channel", "varchar"),
                                      ("ALTER TABLE ext_schema MODIFY COLUMN amount_cents BIGINT", "amount_cents", "bigint")):
    show_sql("Schema Change",statement)
    lab.execute(statement)
    deadline = time.monotonic()+120
    while True:
        with lab.connection.cursor() as cursor:
            cursor.execute("SHOW ALTER TABLE COLUMN WHERE TableName = 'ext_schema' ORDER BY CreateTime DESC LIMIT 1")
            fields = [c[0] for c in cursor.description]
            jobs = [dict(zip(fields,row)) for row in cursor.fetchall()]
        types = {row[0]: row[1].lower() for row in lab.query("DESC ext_schema")}
        if types.get(column, "").startswith(target_type) and (not jobs or jobs[0]["State"] == "FINISHED"):
            break
        if jobs and jobs[0]["State"] == "CANCELLED":
            raise RuntimeError(str(jobs))
        if time.monotonic()>=deadline:
            raise TimeoutError(str(jobs))
        time.sleep(1)
    lab.sql("SHOW ALTER TABLE COLUMN WHERE TableName = 'ext_schema'")
    expect(lab.query("SELECT order_id,amount_cents FROM ext_schema ORDER BY order_id"), expected)
expect(lab.query("SELECT COUNT(*),COUNT(order_channel) FROM ext_schema"), [(10,0)])
lab.sql("DESC ext_schema")

## 2. Load Deletion Markers and Old-Version Replay

Use an independent three-column table with Unique Key + Sequence. The request maps operation types to deletion markers using MERGE and a DELETE ON condition.
First load CREATED v1, then delete at v2, replay v1, and finally explicitly write the new version PAID v3.
The validation sequence is 1 → 0 → 0 → 1 rows; rebuilding with a new version is an explicit new event, not random resurrection of an old record.
Invisibility to ordinary queries does not mean immediate disk release; the business refund table is not changed here.

In [ ]:
import os
import requests
from uuid import uuid4
lab.execute("DROP TABLE IF EXISTS ext_delete_sign")
lab.execute('''CREATE TABLE ext_delete_sign (
order_id BIGINT NOT NULL,status VARCHAR(20) NOT NULL,event_version BIGINT NOT NULL
) UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES("replication_num"="1","enable_unique_key_merge_on_write"="true","function_column.sequence_col"="event_version")''')
endpoint = os.environ["DW_BE_HTTP_URL"].rstrip("/")
for payload, expected in (("900001,CREATED,1,UPSERT\n",[(900001,"CREATED",1)]),
                          ("900001,CREATED,2,DELETE\n",[]),
                          ("900001,CREATED,1,UPSERT\n",[]),
                          ("900001,PAID,3,UPSERT\n",[(900001,"PAID",3)])):
    headers = {"label":"ext_delete_"+uuid4().hex,"format":"csv","column_separator":",",
               "columns":"order_id,status,event_version,op","merge_type":"MERGE","delete":"op='DELETE'",
               "group_commit":"off_mode","strict_mode":"true","max_filter_ratio":"0"}
    response = requests.put(f"{endpoint}/api/{lab.database}/ext_delete_sign/_stream_load",
                            auth=(lab.user,lab.password),headers=headers,data=payload.encode(),
                            allow_redirects=False,timeout=120)
    response.raise_for_status()
    result = response.json()
    show_response(result, title="Delete-sign Stream Load response")
    expect(result["Status"],"Success")
    expect(lab.query("SELECT order_id,status,event_version FROM ext_delete_sign ORDER BY order_id"),expected)

## 3. The Same Event ID, but Different Content

Unique Key(event_id) can overwrite records with the same ID but does not automatically detect content conflicts. First place raw candidate events in an independent staging table.
Identical content still satisfies the idempotency contract; different content for the same ID must stop processing and must not be written directly to the main history or current table.
This example keeps only three business fields: order_id/status/version. Production rules should cover the complete normalized business payload.
This single-writer check-then-write demonstration provides no multi-writer transaction guarantees.

In [ ]:
from dw_course.runtime import expected_failure
lab.execute("DROP TABLE IF EXISTS ext_event_stage")
lab.execute('''CREATE TABLE ext_event_stage (
event_id VARCHAR(32),order_id BIGINT,status VARCHAR(20),event_version BIGINT
) DUPLICATE KEY(event_id) DISTRIBUTED BY HASH(event_id) BUCKETS 1 PROPERTIES("replication_num"="1")''')
lab.execute("INSERT INTO ext_event_stage VALUES ('CONFLICT_DEMO',900001,'PAID',2),('CONFLICT_DEMO',900001,'PAID',2)")
conflicts = """SELECT event_id FROM (
SELECT DISTINCT event_id,order_id,status,event_version FROM ext_event_stage
) v GROUP BY event_id HAVING COUNT(*)>1 ORDER BY event_id"""
expect(lab.query(conflicts),[])
lab.execute("INSERT INTO ext_event_stage VALUES ('CONFLICT_DEMO',900001,'CANCELLED',2)")
with expected_failure("Event content conflict", "Different content for the same ID detected; stop subsequent business writes"):
    expect(lab.query(conflicts),[])
expect(lab.query(conflicts),[("CONFLICT_DEMO",)])
lab.sql("SELECT * FROM ext_event_stage ORDER BY event_id,status")

## 4. Independent Explanation and Troubleshooting

Explain why the channel column can be NULL and which downstream SELECT * statements or positional writes are affected by the new column.
Explain why deletion version 2 blocks old version 1 and why an explicit new version 3 can appear again.
Try changing only the order ID in a staged record and explain why conflict detection still finds the error.
For Schema Change failures, inspect SHOW ALTER TABLE COLUMN; for load failures, inspect the response and ErrorURL; for conflicts, preserve all candidate records rather than overwriting away the evidence.

References: [Schema Change](https://doris.apache.org/docs/4.x/table-design/schema-change/),
[Updates and deletion](https://doris.apache.org/docs/4.x/data-operate/update/update-overview/).

In [ ]:
lab.close()